In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import scipy
import xskillscore as xs
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs

import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

import albedo_functions as af

In [ ]:
plt.rcParams.update({
    'font.size': 20,          # Font di base
    'axes.titlesize': 20,     # Titolo del grafico
    'axes.labelsize': 20,     # Etichette degli assi
    'xtick.labelsize': 20,    # Etichette sull'asse x
    'ytick.labelsize': 20,    # Etichette sull'asse y
    'legend.fontsize': 20,    # Font della legenda
})

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
BASE_PATH = f'{POST_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'

In [ ]:
def plots(season, var):
    ctrl = xr.open_dataset(f'{SAVE_PATH}{exp_ctrl}_{var}_{season}_std_1999.nc')['std']
    sens = xr.open_dataset(f'{SAVE_PATH}{exp_sens}_{var}_{season}_std_1999.nc')['std']

    p_value = xr.open_dataset(f'{SAVE_PATH}delta_{var}_{season}_std_p_1999.nc')['p']

    levels=[0,0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09]
    
    title = f'a) DCPP-CTRL'
    af.map_plot(af.fix_longitude(ctrl), [], levels=levels, title=title, cmap='Greens', antartica=False)
    plt.savefig(f'{FIG_DIR}/a1ua_{var}_{season}_std_1999.png', dpi=600, bbox_inches = 'tight')

    title = f'b) DCPP-SENS'
    af.map_plot(af.fix_longitude(sens), [], levels=levels, title=title, cmap='Greens', antartica=False)
    plt.savefig(f'{FIG_DIR}/a52o_{var}_{season}_std_1999.png', dpi=600, bbox_inches = 'tight')

    levels=[-0.6,-0.5,-0.4,-0.3,-0.2,-0.1,0,0.1,0.2,0.3,0.4,0.5,0.6]
    title=f'c) DCPP-SENS minus DCPP-CTRL'
    af.map_plot(af.fix_longitude(sens - ctrl), af.fix_longitude(p_value), levels=levels,title=title, cmap='BrBG', antartica=False)
    plt.savefig(f'{FIG_DIR}/delta_{var}_{season}_std_1999.png', dpi=600, bbox_inches = 'tight')

    import matplotlib.image as mpimg
    img1 = mpimg.imread(f'{FIG_DIR}/a1ua_{var}_{season}_std_1999.png')
    img2 = mpimg.imread(f'{FIG_DIR}/a52o_{var}_{season}_std_1999.png')
    img3 = mpimg.imread(f'{FIG_DIR}/delta_{var}_{season}_std_1999.png')
    af.triple_figure(img1, img2, img3, f'{FIG_DIR}/fig1_{var}_{season}_1999.png')

In [ ]:
%%time
seasons=['DJF', 'JJA', 'MAM', 'SON']
futures=[]
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures += [executor.submit(plots, i, 'cvh') for i in seasons]
    futures += [executor.submit(plots, i, 'cvl') for i in seasons]

    # Wait for all tasks to complete
    # progresso: stampa ogni task man mano che termina
    for _i, _f in enumerate(concurrent.futures.as_completed(futures), 1):
        _f.result()
        print(f"  completato {_i}/{len(futures)}", flush=True)